In [8]:
import os
import torch
import gc

from ultralytics import YOLO, RTDETR

In [ ]:
ADD_DATA_FOLDS_DIR = 'add_data_folds'
NUM_FOLDS = 2

EPOCHS_YOLO = 15
EPOCHS_DETR = 20

BATCH_SIZE = 8
NUM_CLASSES = 2
IMG_SIZE = 768

BASE_RUNS_DIR = 'runs/detect/runs/ensemble_training'

print(f"Configuration: {NUM_FOLDS} folds, YOLO={EPOCHS_YOLO} epochs, RT-DETR={EPOCHS_DETR} epochs")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Data: {ADD_DATA_FOLDS_DIR}/fold_N/")

Configuration: 2 folds, YOLO=15 epochs, RT-DETR=20 epochs
Device: cuda
Data: add_data_folds/fold_N/


In [ ]:
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def on_epoch_end_callback(*args, **kwargs):
    free_memory()

In [ ]:
def finetune_yolo11_model(fold_idx, fold_dir, weights_path):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"\n{'='*60}")
    print(f" Fine-tuning YOLO11m | Fold {fold_idx} | {EPOCHS_YOLO} epochs")
    print(f" Weights: {weights_path}")
    print(f"{'='*60}\n")

    yolo_model = YOLO(weights_path)
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)

    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        patience=10,
        project='runs/finetune_add_data',
        name=f'yolo11m_fold_{fold_idx}_finetuned',
        device=0,
        workers=4,
        augment=True,

        box=7.5,
        cls=1.0,
        dfl=1.5,

        cos_lr=True,

        fliplr=0.5,
        flipud=0.0,
        degrees=20.0,
        translate=0.15,
        scale=0.2,
        shear=10.0,
        perspective=0.0001,

        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,

        mosaic=0.5,
        mixup=0.15,
        cutmix=0.1,
        copy_paste=0.2,
        erasing=0.4,

        close_mosaic=5,
    )

    del yolo_model
    free_memory()
    print(f" YOLO11m Fold {fold_idx} fine-tuning completed\n")

In [ ]:
def finetune_yolo26_model(fold_idx, fold_dir, weights_path):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"\n{'='*60}")
    print(f" Fine-tuning YOLO26m | Fold {fold_idx} | {EPOCHS_YOLO} epochs")
    print(f" Weights: {weights_path}")
    print(f"{'='*60}\n")

    yolo_model = YOLO(weights_path)
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)

    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        patience=10,
        project='runs/finetune_add_data',
        name=f'yolo26m_fold_{fold_idx}_finetuned',
        device=0,
        workers=4,
        augment=True,

        box=7.5,
        cls=1.0,
        dfl=1.5,

        cos_lr=True,

        fliplr=0.5,
        flipud=0.0,
        degrees=20.0,
        translate=0.15,
        scale=0.2,
        shear=10.0,
        perspective=0.0001,

        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,

        mosaic=0.5,
        mixup=0.15,
        cutmix=0.1,
        copy_paste=0.2,
        erasing=0.4,

        close_mosaic=5,
    )

    del yolo_model
    free_memory()
    print(f" YOLO26m Fold {fold_idx} fine-tuning completed\n")

In [ ]:
def finetune_rtdetr_model(fold_idx, fold_dir, weights_path):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"\n{'='*60}")
    print(f" Fine-tuning RT-DETR | Fold {fold_idx} | {EPOCHS_DETR} epochs")
    print(f" Weights: {weights_path}")
    print(f"{'='*60}\n")

    detr_model = RTDETR(weights_path)
    detr_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)

    detr_model.train(
        data=yaml_path,
        epochs=EPOCHS_DETR,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=10,
        project='runs/finetune_add_data',
        name=f'rtdetr_fold_{fold_idx}_finetuned',
        device=0,
        workers=4,
        augment=True,

        cos_lr=True,

        fliplr=0.5,
        flipud=0.0,
        degrees=20.0,
        translate=0.15,
        scale=0.2,
        shear=10.0,
        perspective=0.0001,

        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,

        mosaic=0.5,
        mixup=0.15,
        cutmix=0.1,
        copy_paste=0.2,
        erasing=0.4,
    )

    del detr_model
    free_memory()
    print(f" RT-DETR Fold {fold_idx} fine-tuning completed\n")

In [ ]:
def main():
    """Fine-tune all models on their corresponding add_data folds."""

    for fold_idx in range(1, NUM_FOLDS + 1):
        fold_dir = os.path.join(ADD_DATA_FOLDS_DIR, f'fold{fold_idx}')

        if not os.path.exists(fold_dir):
            print(f"Warning: {fold_dir} does not exist. Skipping fold {fold_idx}.")
            continue

        yaml_path = os.path.join(fold_dir, 'data.yaml')
        if not os.path.exists(yaml_path):
            print(f"Warning: {yaml_path} not found. Please run create_finetune_yaml.py first.")
            print(f"Expected structure: {fold_dir}/train/images/ and {fold_dir}/val/images/")
            continue

        print(f"\n{'#'*60}")
        print(f"# FOLD {fold_idx} FINE-TUNING")
        print(f"{'#'*60}")

        yolo11_weights = os.path.join(BASE_RUNS_DIR, f'yolo11m_fold_{fold_idx}', 'weights', 'best.pt')
        yolo26_weights = os.path.join(BASE_RUNS_DIR, f'yolo26m_fold_{fold_idx}', 'weights', 'best.pt')
        rtdetr_weights = os.path.join(BASE_RUNS_DIR, f'rtdetr_fold_{fold_idx}', 'weights', 'best.pt')

        if os.path.exists(yolo11_weights):
            finetune_yolo11_model(fold_idx, fold_dir, yolo11_weights)
        else:
            print(f"Warning: YOLO11m weights not found: {yolo11_weights}")

        if os.path.exists(yolo26_weights):
            finetune_yolo26_model(fold_idx, fold_dir, yolo26_weights)
        else:
            print(f"Warning: YOLO26m weights not found: {yolo26_weights}")

        if os.path.exists(rtdetr_weights):
            finetune_rtdetr_model(fold_idx, fold_dir, rtdetr_weights)
        else:
            print(f"Warning: RT-DETR weights not found: {rtdetr_weights}")

        free_memory()

    print("\n" + "="*60)
    print(" ALL FINE-TUNING COMPLETED!")
    print("="*60)
    print(f"\nFine-tuned models saved to: runs/finetune_add_data/")


if __name__ == '__main__':
    main()


############################################################
# FOLD 1 FINE-TUNING
############################################################

 Fine-tuning YOLO11m | Fold 1 | 15 epochs
 Weights: runs/detect/runs/ensemble_training\yolo11m_fold_1\weights\best.pt

New https://pypi.org/project/ultralytics/8.4.23 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=1.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.1, data=d:\LabsMIET\ML-Detection-object\add_data_folds\fold1\data.yaml, degrees=20.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscr

d:\LabsMIET\ML-Detection-object\.venv\Lib\site-packages\torch\autograd\graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      6.64G     0.5664     0.6634     0.3327         13        640: 100% ━━━━━━━━━━━━ 126/126 1.3it/s 1:360.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 2.5it/s 19.4s0.4s
                   all        782       4397      0.857      0.736      0.812      0.497

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\LabsMIET\ML-Detection-object\.venv\Lib\site-packages\torch\autograd\graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/20       6.8G     0.5971     0.7008      0.342         20        640: 100% ━━━━━━━━━━━━ 126/126 1.4it/s 1:290.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 2.8it/s 17.6s0.4s
                   all        782       4397      0.721       0.62      0.676      0.359

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      18/20       6.7G     0.3777     0.4892     0.1994         23        640: 100% ━━━━━━━━━━━━ 126/126 1.6it/s 1:210.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.1it/s 16.0s0.3s
                   all        782       4212      0.801      0.608       0.63      0.286

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\LabsMIET\ML-Detection-object\.venv\Lib\site-packages\torch\autograd\graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      19/20      6.75G     0.3696     0.4981     0.1993         21        640: 100% ━━━━━━━━━━━━ 126/126 1.6it/s 1:210.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.1it/s 15.9s0.3s
                   all        782       4212      0.808      0.621      0.646      0.331

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\LabsMIET\ML-Detection-object\.venv\Lib\site-packages\torch\autograd\graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      20/20      6.73G     0.3629     0.4883     0.1936         22        640: 100% ━━━━━━━━━━━━ 126/126 1.6it/s 1:200.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.1it/s 15.9s0.3s
                   all        782       4212      0.791      0.631       0.65      0.322

20 epochs completed in 0.628 hours.
Optimizer stripped from D:\LabsMIET\ML-Detection-object\runs\detect\runs\finetune_add_data\rtdetr_fold_2_finetuned\weights\last.pt, 66.2MB
Optimizer stripped from D:\LabsMIET\ML-Detection-object\runs\detect\runs\finetune_add_data\rtdetr_fold_2_finetuned\weights\best.pt, 66.2MB

Validating D:\LabsMIET\ML-Detection-object\runs\detect\runs\finetune_add_data\rtdetr_fold_2_finetuned\weights\best.pt...
Ultralytics 8.4.21  Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances